In [ ]:
import pandas as pd
import numpy as np
import Bio
from Bio.Restriction import AllEnzymes
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 加载数据

In [ ]:
# Load data files
df_methylation = pd.read_csv('./utils/output/methylation_check.csv')
df_seamless = pd.read_csv('./utils/output/restriction_enzyme_seamless_insert.csv')
df_silent = pd.read_csv('./utils/output/restriction_enzyme_slient_mutation.csv')

print(f"Methylation check: {df_methylation.shape}")
print(f"Seamless insert: {df_seamless.shape}")
print(f"Silent mutation: {df_silent.shape}")

## 辅助函数

In [ ]:
def check_methylation_compatible(enzyme_name, df_methylation):
    """Check if enzyme is not sensitive to 6mA/5mC methylation (DH5α compatible)"""
    dh5a_compatible = set(
        df_methylation.loc[~df_methylation['6mA_5mC_sensitive'], 'enzyme'].dropna().tolist() +
        df_methylation.loc[~df_methylation['6mA_5mC_sensitive'], 'prototype'].dropna().tolist()
    )
    return enzyme_name in dh5a_compatible

def get_enzyme_info(enzyme_name):
    """Get detailed information about an enzyme"""
    try:
        enzyme = getattr(Bio.Restriction, enzyme_name)
        site = str(enzyme.site)
        site_length = len(enzyme.site)
        fst5 = enzyme.fst5
        fst3 = enzyme.fst3
        ovhg = enzyme.ovhg
        
        # 判断切割类型
        is_type_iis = fst5 > site_length or fst3 > 0  # fst3通常是负数，正数表示切在外面
        cuts_outside = fst5 > site_length
        
        return {
            'site': site,
            'site_length': site_length,
            'fst5': fst5,
            'fst3': fst3,
            'ovhg': ovhg,
            'cuts_outside_5': cuts_outside,
            'is_type_iis': is_type_iis
        }
    except Exception as e:
        return None

## Site I 候选酶分析（Seamless Insert）

In [ ]:
# Get Site I candidates
site_i_enzymes = df_seamless['name'].unique()
print(f"Total enzymes in seamless insert file: {len(site_i_enzymes)}")

# Filter by methylation
site_i_filtered = [e for e in site_i_enzymes if check_methylation_compatible(e, df_methylation)]
print(f"After methylation filter: {len(site_i_filtered)}")

# Get detailed info
site_i_info = []
for enzyme in site_i_filtered:
    info = get_enzyme_info(enzyme)
    if info:
        info['enzyme'] = enzyme
        site_i_info.append(info)

df_site_i = pd.DataFrame(site_i_info)
print(f"\nSite I candidates with enzyme info: {len(df_site_i)}")
print(f"\nCut types distribution:")
print(df_site_i['is_type_iis'].value_counts())
print(df_site_i['cuts_outside_5'].value_counts())

In [ ]:
# Display Site I candidates
print("\n=" * 80)
print("SITE I CANDIDATES (Seamless Insert)")
print("=" * 80)
display(df_site_i.sort_values('enzyme'))

## Site II 候选酶分析（Silent Mutation）

In [ ]:
# Get Site II candidates
site_ii_enzymes = df_silent['name'].unique()
print(f"Total enzymes in silent mutation file: {len(site_ii_enzymes)}")

# Filter by methylation
site_ii_filtered = [e for e in site_ii_enzymes if check_methylation_compatible(e, df_methylation)]
print(f"After methylation filter: {len(site_ii_filtered)}")

# Get detailed info
site_ii_info = []
for enzyme in site_ii_filtered:
    info = get_enzyme_info(enzyme)
    if info:
        info['enzyme'] = enzyme
        site_ii_info.append(info)

df_site_ii = pd.DataFrame(site_ii_info)
print(f"\nSite II candidates with enzyme info: {len(df_site_ii)}")
print(f"\nCut types distribution:")
print(df_site_ii['is_type_iis'].value_counts())
print(df_site_ii['cuts_outside_5'].value_counts())

In [ ]:
# Display Site II candidates
print("\n=" * 80)
print("SITE II CANDIDATES (Silent Mutation)")
print("=" * 80)
display(df_site_ii.sort_values('enzyme'))

## Site III 候选酶分析

**需要确定：Site III应该满足什么条件？**

可能的理解：
1. Type IIS酶（fst5 > site_length）- 切割在识别位点外
2. 与Site II不同的酶，但产生相同overhang
3. 其他特殊条件？

让我们检查所有可能的候选：

In [ ]:
# Site III也来自silent mutation，但需要特殊条件
print("检查不同切割类型的酶数量：")
print(f"\nTotal silent mutation enzymes: {len(df_site_ii)}")
print(f"  - cuts_outside_5=True (fst5 > site_length): {df_site_ii['cuts_outside_5'].sum()}")
print(f"  - cuts_outside_5=False (fst5 <= site_length): {(~df_site_ii['cuts_outside_5']).sum()}")
print(f"  - is_type_iis=True: {df_site_ii['is_type_iis'].sum()}")
print(f"  - is_type_iis=False: {(~df_site_ii['is_type_iis']).sum()}")

In [ ]:
# 检查fst5和fst3的分布
print("\nfst5 distribution:")
print(df_site_ii['fst5'].value_counts().sort_index())

print("\nfst3 distribution:")
print(df_site_ii['fst3'].value_counts().sort_index())

In [ ]:
# 显示几个典型例子
print("\n=" * 80)
print("典型例子")
print("=" * 80)

print("\n1. 常规酶 (fst5 <= site_length):")
display(df_site_ii[~df_site_ii['cuts_outside_5']].head(10))

print("\n2. Type IIS酶 (fst5 > site_length):")
type_iis_examples = df_site_ii[df_site_ii['cuts_outside_5']]
if len(type_iis_examples) > 0:
    display(type_iis_examples.head(10))
else:
    print("没有找到Type IIS酶！")

## Overhang分析

Site II和Site III需要产生相同的overhang用于连接重复单元

In [ ]:
print("Overhang分布（Silent Mutation酶）:")
print(df_site_ii['ovhg'].value_counts().sort_index())

print("\n每个overhang类型的酶数量：")
for ovhg in sorted(df_site_ii['ovhg'].unique()):
    enzymes_with_ovhg = df_site_ii[df_site_ii['ovhg'] == ovhg]
    print(f"  ovhg={ovhg:>3}: {len(enzymes_with_ovhg)} enzymes")
    if len(enzymes_with_ovhg) > 1:
        print(f"      可用于Site II/III配对: {list(enzymes_with_ovhg['enzyme'].values[:5])}...")

## 切割位点详细分析

理解fst5和fst3的含义：
- fst5: 5'链上的切割位置（从5'端开始计数）
- fst3: 3'链上的切割位置（通常是负数，从3'端开始计数）

示例：
```
EcoRI: GAATTC (site_length=6)
       G^AATTC  (fst5=1, 在第1个碱基后切割)
       CTTAA^G  (fst3=-1, 从末尾倒数第1个碱基前切割)
```

In [ ]:
# 选择几个代表性的酶展示切割模式
examples = [
    'EcoRI',   # 经典常规酶
    'BamHI',   # 经典常规酶
    'BsaI',    # 经典Type IIS
    'PstI',    # 常规酶
    'SmaI',    # 平末端
]

print("\n代表性酶的切割模式：")
print("=" * 80)
for enzyme_name in examples:
    info = get_enzyme_info(enzyme_name)
    if info:
        print(f"\n{enzyme_name}:")
        print(f"  Recognition site: {info['site']} (length={info['site_length']})")
        print(f"  Cut positions: fst5={info['fst5']}, fst3={info['fst3']}")
        print(f"  Overhang: {info['ovhg']}")
        print(f"  Cuts outside 5' end: {info['cuts_outside_5']}")
        print(f"  Type IIS: {info['is_type_iis']}")

## 总结与建议

**请根据上面的信息回答以下问题：**

1. Site I应该选择哪些类型的酶？（目前：seamless insert中的酶，methylation compatible）

2. Site II应该选择哪些类型的酶？（目前：silent mutation中的酶，methylation compatible）

3. **Site III应该选择哪些类型的酶？请明确指定判断条件：**
   - 选项A：Type IIS酶（fst5 > site_length）
   - 选项B：与Site II相同pool，但不同酶，相同overhang
   - 选项C：其他条件？

4. "识别位点和切割位点不重叠"具体指什么？

In [ ]:
# 保存结果供参考
df_site_i.to_csv('output/site_i_candidates_detail.csv', index=False)
df_site_ii.to_csv('output/site_ii_candidates_detail.csv', index=False)

print("\n候选酶详细信息已保存：")
print("  - output/site_i_candidates_detail.csv")
print("  - output/site_ii_candidates_detail.csv")